# IQL + Flat Observations - Multi-Agent Coverage

This notebook trains Independent Q-Learning (IQL) agents with flat 64-dimensional observations on the multi-agent coverage task.

**Features:**
- Real-time training monitoring
- Live visualization of coverage heatmaps
- Training curves and metrics
- Agent trajectory analysis
- Model saving and evaluation

**Training Time:** ~2-3 hours on Colab GPU

## 1. Setup and Installation

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install dependencies
!pip install torch torchvision pygame pyyaml tensorboard sacred numpy gym
!pip install matplotlib seaborn pandas

In [ ]:
# Clone the repository
!git clone https://github.com/ayyan-k98/ind-q.git
%cd ind-q

In [ ]:
# Clone EPyMARL
!git clone https://github.com/uoe-agents/epymarl.git

# Setup coverage environment in EPyMARL
!python epymarl_integration/setup_coverage.py ./epymarl

## 2. Import Libraries and Setup Monitoring

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import clear_output, display
import pandas as pd
from datetime import datetime
import os
import sys
import time
import json

# Add EPyMARL to path
sys.path.insert(0, '/content/ind-q/epymarl/src')

# Setup plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

In [ ]:
# Create directories for outputs
!mkdir -p /content/ind-q/results/plots
!mkdir -p /content/ind-q/results/models
!mkdir -p /content/ind-q/results/videos

## 3. Training Configuration

In [ ]:
# Training configuration
CONFIG = {
    # Algorithm
    'algorithm': 'iql',
    'env_config': 'coverage_flat',
    
    # Training
    'n_episodes': 5000,
    'batch_size': 32,
    'learning_rate': 0.0005,
    'gamma': 0.99,
    'buffer_size': 5000,
    
    # Exploration
    'epsilon_start': 1.0,
    'epsilon_end': 0.05,
    'epsilon_anneal_steps': 50000,
    
    # Monitoring
    'log_interval': 10,        # Log every N episodes
    'plot_interval': 50,       # Update plots every N episodes
    'save_interval': 500,      # Save model every N episodes
    
    # Environment
    'grid_size': 20,
    'n_agents': 4,
    'max_steps': 200,
    
    # Device
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

print(f"Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## 4. Initialize Monitoring Dashboard

In [ ]:
class TrainingMonitor:
    """Real-time training monitoring and visualization"""
    
    def __init__(self):
        self.metrics = {
            'episode': [],
            'return': [],
            'coverage': [],
            'epsilon': [],
            'loss': [],
            'steps': [],
            'time': []
        }
        self.start_time = time.time()
        
    def log(self, episode, return_val, coverage, epsilon, loss, steps):
        """Log metrics for an episode"""
        self.metrics['episode'].append(episode)
        self.metrics['return'].append(return_val)
        self.metrics['coverage'].append(coverage)
        self.metrics['epsilon'].append(epsilon)
        self.metrics['loss'].append(loss)
        self.metrics['steps'].append(steps)
        self.metrics['time'].append(time.time() - self.start_time)
    
    def plot(self, save_path=None):
        """Plot training curves"""
        if len(self.metrics['episode']) < 2:
            return
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle(f'IQL Training Progress (Episode {self.metrics["episode"][-1]})', 
                     fontsize=16, fontweight='bold')
        
        # Returns
        ax = axes[0, 0]
        ax.plot(self.metrics['episode'], self.metrics['return'], alpha=0.3, color='blue')
        if len(self.metrics['return']) > 20:
            window = min(50, len(self.metrics['return']) // 10)
            smoothed = pd.Series(self.metrics['return']).rolling(window=window).mean()
            ax.plot(self.metrics['episode'], smoothed, color='blue', linewidth=2)
        ax.set_xlabel('Episode')
        ax.set_ylabel('Return')
        ax.set_title('Episode Return')
        ax.grid(True, alpha=0.3)
        
        # Coverage
        ax = axes[0, 1]
        ax.plot(self.metrics['episode'], self.metrics['coverage'], alpha=0.3, color='green')
        if len(self.metrics['coverage']) > 20:
            window = min(50, len(self.metrics['coverage']) // 10)
            smoothed = pd.Series(self.metrics['coverage']).rolling(window=window).mean()
            ax.plot(self.metrics['episode'], smoothed, color='green', linewidth=2)
        ax.set_xlabel('Episode')
        ax.set_ylabel('Coverage %')
        ax.set_title('Final Coverage')
        ax.grid(True, alpha=0.3)
        
        # Epsilon
        ax = axes[0, 2]
        ax.plot(self.metrics['episode'], self.metrics['epsilon'], color='orange', linewidth=2)
        ax.set_xlabel('Episode')
        ax.set_ylabel('Epsilon')
        ax.set_title('Exploration Rate')
        ax.grid(True, alpha=0.3)
        
        # Loss
        ax = axes[1, 0]
        valid_losses = [(e, l) for e, l in zip(self.metrics['episode'], self.metrics['loss']) if l is not None]
        if valid_losses:
            episodes, losses = zip(*valid_losses)
            ax.plot(episodes, losses, alpha=0.3, color='red')
            if len(losses) > 20:
                window = min(50, len(losses) // 10)
                smoothed = pd.Series(losses).rolling(window=window).mean()
                ax.plot(episodes, smoothed, color='red', linewidth=2)
        ax.set_xlabel('Episode')
        ax.set_ylabel('Loss')
        ax.set_title('Training Loss')
        ax.set_yscale('log')
        ax.grid(True, alpha=0.3)
        
        # Episode Length
        ax = axes[1, 1]
        ax.plot(self.metrics['episode'], self.metrics['steps'], alpha=0.3, color='purple')
        if len(self.metrics['steps']) > 20:
            window = min(50, len(self.metrics['steps']) // 10)
            smoothed = pd.Series(self.metrics['steps']).rolling(window=window).mean()
            ax.plot(self.metrics['episode'], smoothed, color='purple', linewidth=2)
        ax.set_xlabel('Episode')
        ax.set_ylabel('Steps')
        ax.set_title('Episode Length')
        ax.grid(True, alpha=0.3)
        
        # Statistics
        ax = axes[1, 2]
        ax.axis('off')
        stats_text = f"""
        Training Statistics:
        
        Episodes: {self.metrics['episode'][-1]}
        
        Return (last 100):
          Mean: {np.mean(self.metrics['return'][-100:]):.2f}
          Max: {np.max(self.metrics['return'][-100:]):.2f}
        
        Coverage (last 100):
          Mean: {np.mean(self.metrics['coverage'][-100:]):.1f}%
          Max: {np.max(self.metrics['coverage'][-100:]):.1f}%
        
        Epsilon: {self.metrics['epsilon'][-1]:.3f}
        
        Time: {self.metrics['time'][-1]/60:.1f} min
        """
        ax.text(0.1, 0.5, stats_text, fontsize=12, family='monospace',
                verticalalignment='center')
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
        
        plt.show()
    
    def save_metrics(self, path):
        """Save metrics to file"""
        df = pd.DataFrame(self.metrics)
        df.to_csv(path, index=False)
        print(f"Metrics saved to {path}")

# Initialize monitor
monitor = TrainingMonitor()
print("Training monitor initialized!")

## 5. Start Training

This cell runs the training with EPyMARL. You can monitor progress in real-time.

In [ ]:
# Start training with EPyMARL
%cd /content/ind-q/epymarl

# Create unique experiment name
exp_name = f"iql_flat_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Training command
!python src/main.py \
    --config=iql_flat \
    --env-config=coverage_flat \
    with \
    name="{exp_name}" \
    t_max=1000000 \
    test_interval=10000 \
    log_interval=1000 \
    runner_log_interval=1000 \
    learner_log_interval=1000 \
    save_model=True \
    save_model_interval=100000 \
    checkpoint_path="results/models/{exp_name}"

## 6. Monitor Training with TensorBoard

In [ ]:
# Load TensorBoard
%load_ext tensorboard
%tensorboard --logdir /content/ind-q/epymarl/results/tb_logs

## 7. Parse and Visualize Training Logs

In [ ]:
import glob
import re

def parse_sacred_logs(log_dir):
    """Parse Sacred logging output"""
    metrics = {
        'timestep': [],
        'return_mean': [],
        'return_std': [],
        'test_return_mean': [],
        'test_return_std': [],
        'coverage_mean': [],
        'epsilon': [],
        'loss': []
    }
    
    # Find the latest experiment
    exp_dirs = sorted(glob.glob(f"{log_dir}/sacred/*"), key=os.path.getmtime)
    if not exp_dirs:
        print("No experiment logs found")
        return None
    
    latest_exp = exp_dirs[-1]
    info_file = os.path.join(latest_exp, 'info.json')
    
    if os.path.exists(info_file):
        with open(info_file, 'r') as f:
            data = json.load(f)
            print(f"Experiment: {data.get('name', 'Unknown')}")
            print(f"Status: {data.get('status', 'Unknown')}")
    
    return metrics

# Parse logs
log_dir = "/content/ind-q/epymarl/results"
metrics = parse_sacred_logs(log_dir)

## 8. Evaluation and Visualization

In [ ]:
# Load trained model and evaluate
%cd /content/ind-q/epymarl

# Find the latest checkpoint
checkpoint_dirs = sorted(glob.glob("results/models/*"), key=os.path.getmtime)
if checkpoint_dirs:
    latest_checkpoint = checkpoint_dirs[-1]
    print(f"Loading checkpoint from: {latest_checkpoint}")
    
    # Run evaluation
    !python src/main.py \
        --config=iql_flat \
        --env-config=coverage_flat \
        with \
        checkpoint_path="{latest_checkpoint}" \
        evaluate=True \
        test_nepisode=20 \
        render=False
else:
    print("No checkpoints found. Train the model first.")

## 9. Custom Visualization: Coverage Heatmaps

In [ ]:
# Add project to path
import sys
sys.path.insert(0, '/content/ind-q')

from epymarl_integration.coverage_env import CoverageEnv
import numpy as np

# Create environment
env = CoverageEnv(
    grid_size=20,
    n_agents=4,
    max_steps=200,
    obstacle_density=0.1
)

def visualize_episode(env, n_steps=200):
    """Visualize a single episode with coverage heatmap"""
    env.reset()
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    for step in range(n_steps):
        # Take random actions (replace with trained agent actions)
        actions = env.get_avail_actions()
        selected_actions = []
        for agent_actions in actions:
            available = np.where(agent_actions)[0]
            selected_actions.append(np.random.choice(available))
        
        reward, done, info = env.step(selected_actions)
        
        # Update visualization every 50 steps
        if step % 50 == 0 or done:
            clear_output(wait=True)
            
            # Coverage heatmap
            ax = axes[0]
            ax.clear()
            im = ax.imshow(env.coverage_grid, cmap='YlGn', vmin=0, vmax=1)
            # Plot agents
            for agent in env.agents:
                ax.plot(agent.y, agent.x, 'ro', markersize=10)
            ax.set_title(f'Coverage Heatmap (Step {step})')
            ax.set_xlabel('Y')
            ax.set_ylabel('X')
            plt.colorbar(im, ax=ax, label='Coverage')
            
            # Obstacle map
            ax = axes[1]
            ax.clear()
            ax.imshow(env.obstacle_grid, cmap='binary')
            ax.set_title('Obstacle Map')
            ax.set_xlabel('Y')
            ax.set_ylabel('X')
            
            # Agent trajectories
            ax = axes[2]
            ax.clear()
            ax.imshow(env.obstacle_grid, cmap='Greys', alpha=0.3)
            colors = ['red', 'blue', 'green', 'orange']
            for i, agent in enumerate(env.agents):
                if hasattr(agent, 'trajectory'):
                    traj = np.array(agent.trajectory)
                    ax.plot(traj[:, 1], traj[:, 0], 
                           color=colors[i % len(colors)], 
                           alpha=0.6, linewidth=2,
                           label=f'Agent {i}')
            ax.set_title('Agent Trajectories')
            ax.set_xlabel('Y')
            ax.set_ylabel('X')
            ax.legend()
            
            plt.tight_layout()
            plt.show()
            
        if done:
            break
    
    coverage_pct = (env.coverage_grid > 0).sum() / (env.grid_size ** 2) * 100
    print(f"\nFinal Coverage: {coverage_pct:.1f}%")
    print(f"Total Steps: {step + 1}")

# Run visualization
visualize_episode(env)

## 10. Performance Analysis

In [ ]:
def analyze_performance(env, n_episodes=20):
    """Analyze performance over multiple episodes"""
    coverage_results = []
    episode_lengths = []
    returns = []
    
    for ep in range(n_episodes):
        env.reset()
        episode_return = 0
        
        for step in range(env.max_steps):
            # Random policy (replace with trained agent)
            actions = env.get_avail_actions()
            selected_actions = []
            for agent_actions in actions:
                available = np.where(agent_actions)[0]
                selected_actions.append(np.random.choice(available))
            
            reward, done, info = env.step(selected_actions)
            episode_return += reward
            
            if done:
                break
        
        coverage_pct = (env.coverage_grid > 0).sum() / (env.grid_size ** 2) * 100
        coverage_results.append(coverage_pct)
        episode_lengths.append(step + 1)
        returns.append(episode_return)
    
    # Plot results
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Coverage distribution
    ax = axes[0]
    ax.hist(coverage_results, bins=20, color='green', alpha=0.7, edgecolor='black')
    ax.axvline(np.mean(coverage_results), color='red', linestyle='--', 
               linewidth=2, label=f'Mean: {np.mean(coverage_results):.1f}%')
    ax.set_xlabel('Coverage %')
    ax.set_ylabel('Frequency')
    ax.set_title('Coverage Distribution')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Episode length distribution
    ax = axes[1]
    ax.hist(episode_lengths, bins=20, color='blue', alpha=0.7, edgecolor='black')
    ax.axvline(np.mean(episode_lengths), color='red', linestyle='--',
               linewidth=2, label=f'Mean: {np.mean(episode_lengths):.1f}')
    ax.set_xlabel('Episode Length')
    ax.set_ylabel('Frequency')
    ax.set_title('Episode Length Distribution')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Returns distribution
    ax = axes[2]
    ax.hist(returns, bins=20, color='purple', alpha=0.7, edgecolor='black')
    ax.axvline(np.mean(returns), color='red', linestyle='--',
               linewidth=2, label=f'Mean: {np.mean(returns):.2f}')
    ax.set_xlabel('Return')
    ax.set_ylabel('Frequency')
    ax.set_title('Return Distribution')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('/content/ind-q/results/plots/iql_performance_analysis.png', 
                dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nPerformance Summary (n={n_episodes} episodes):")
    print(f"  Coverage: {np.mean(coverage_results):.1f}% ± {np.std(coverage_results):.1f}%")
    print(f"  Episode Length: {np.mean(episode_lengths):.1f} ± {np.std(episode_lengths):.1f}")
    print(f"  Return: {np.mean(returns):.2f} ± {np.std(returns):.2f}")

# Run analysis
analyze_performance(env, n_episodes=20)

## 11. Save and Download Results

In [ ]:
# Zip results for download
!zip -r /content/iql_flat_results.zip /content/ind-q/results/

# Download the results
from google.colab import files
files.download('/content/iql_flat_results.zip')

print("Results saved and ready for download!")

## 12. Load and Continue Training

In [ ]:
# To continue training from a checkpoint:
%cd /content/ind-q/epymarl

checkpoint_path = "results/models/YOUR_CHECKPOINT_DIR"  # Update this

!python src/main.py \
    --config=iql_flat \
    --env-config=coverage_flat \
    with \
    checkpoint_path="{checkpoint_path}" \
    load_step=0 \
    t_max=2000000